Conversions (legacy aggregate API)
==================================
`bytemaker.conversions` is the original bytemaker record API:
`@dataclass`-decorated classes whose fields are BitTypes, standard-library
`ctypes`, or plain Python types, serialized with
`bytemaker.conversions.aggregate_types.to_bits_aggregate` and rebuilt with
`from_bits_aggregate`. It remains fully supported. For new code, prefer
`bytemaker.Struct`, which compiles the layout once and holds plain Python
values.

`pytypes.py` contains methods and configuration options for automatically converting native Python types
    to bits. This is accomplished through `bytemaker.conversions.pytypes.ConversionConfig`, which controls the
    `bytemaker.conversions.pytypes.bits_to_pytype` and `bytemaker.conversions.pytypes.pytype_to_bits` methods
    for converting single instances of Python types to/from BitVectors.

`ctypes_.py` contains methods for converting Python's `ctypes` data structures to/from BitVector objects, generalized by
    `bytemaker.conversions.ctypes_.bits_to_ctype` and `bytemaker.conversions.ctypes_.ctype_to_bits`

Finally, `aggregate_types.py` combines the above conversions into single methods that handle
instances of `ctypes` types, Python native types, and BitTypes:
`to_bits_aggregate` and `from_bits_aggregate`.


In [1]:
import ctypes
from bytemaker.bittypes import SInt32, Float32, Buffer, UTF8String
from bytemaker.conversions.aggregate_types import to_bits_aggregate, from_bits_aggregate
from dataclasses import dataclass

# Sized buffer/string types are minted on demand:
Buffer4 = Buffer.specialize(4)   # 4 bits
Str8 = UTF8String.of(nbytes=1)   # 8 bits of utf-8, like char

@dataclass
class PyTypeAggregate:
    a: int
    b: float
    c: str  # Char
    d: str  # Char


@dataclass
class CTypeAggregate:
    a: ctypes.c_int32
    b: ctypes.c_float
    c: ctypes.c_char
    d: ctypes.c_char


@dataclass
class YTypeAggregate:
    a: SInt32
    b: Float32
    c: Str8
    d: Str8
    e: Buffer4


@dataclass
class MixedAggregate:
    pytype_aggregate: PyTypeAggregate
    ctype_aggregate: CTypeAggregate
    ytype_aggregate: YTypeAggregate
    hp: int


mixedagg = MixedAggregate(
    PyTypeAggregate(
        512,
        3.14,
        'A',
        'B'
    ),
    CTypeAggregate(
        ctypes.c_int32(512),
        ctypes.c_float(3.14),
        ctypes.c_char('A'.encode('utf-8')),
        ctypes.c_char('B'.encode('utf-8'))
    ),
    YTypeAggregate(
        SInt32(382),
        Float32(3.14),
        Str8('A'),
        Str8('B'),
        Buffer4('0b0101')
    ),
    255
)

mixed_agg_as_bits = to_bits_aggregate(mixedagg)

print(mixed_agg_as_bits.hex(bytes_per_sep=2, sep='_'))

mixedagg.ytype_aggregate.a = SInt32(512)

mixed_agg_as_bits = to_bits_aggregate(mixedagg)
print(mixed_agg_as_bits.hex(bytes_per_sep=2, sep='_'))

mixed_agg_from_bits = from_bits_aggregate(mixed_agg_as_bits, MixedAggregate)
print(mixed_agg_from_bits)


0x0000_0200_4048_f5c3_4142_0000_0200_4048_f5c3_4142_0000_017e_4048_f5c3_4142_5000_000f_f
0x0000_0200_4048_f5c3_4142_0000_0200_4048_f5c3_4142_0000_0200_4048_f5c3_4142_5000_000f_f
MixedAggregate(pytype_aggregate=PyTypeAggregate(a=512, b=3.140000104904175, c='A', d='B'), ctype_aggregate=CTypeAggregate(a=c_long(512), b=c_float(3.140000104904175), c=c_char(b'A'), d=c_char(b'B')), ytype_aggregate=YTypeAggregate(a=SInt32(bits='00000000000000000000001000000000', endianness='big', int_format='twos_complement'), b=Float32(bits='01000000010010001111010111000011', endianness='big'), c=UTF8Stringx1(bits='01000001', endianness='big'), d=UTF8Stringx1(bits='01000010', endianness='big'), e=_Buffer(bits='0101', endianness='big')), hp=255)
